# HYPERVIEW2 Downstream Verification

Notebook przygotowuje srodowisko Colab do walidacji downstream task dla HYPERVIEW2. Wersja bazowa uzywa oficjalnego layoutu EOTDL, jawnego splitu wewnetrznego z `train_gt.csv` oraz gotowych regresorow `scikit-learn` na cechach spektralnych. Sekcja z `RECON_VARIANTS` pozwala pozniej dodac rekonstrukcje z modeli kompresji bez zmiany protokolu regresji.

Najwazniejsze zalozenia protokolu:
- dane wejściowe: `HYPERVIEW2/train/<modality>/*.npz`, etykiety: `HYPERVIEW2/train_gt.csv`,
- targety: `B`, `Cu`, `Zn`, `Fe`, `S`, `Mn`,
- split train/validation jest deterministyczny i zapisywany w wynikach,
- wynik `hyperview_score` to sredni MSE per target znormalizowany przez MSE predykcji sredniej z train setu,
- modele bazowe nie widza danych validation podczas trenowania ani standaryzacji.

## 1. Instalacja zaleznosci

In [ ]:
%pip -q install eotdl scikit-learn pandas numpy tqdm matplotlib joblib

# Opcjonalne modele boostingowe. Wlacz, jesli chcesz porownac LightGBM/CatBoost/XGBoost.
INSTALL_OPTIONAL_BOOSTING = False
if INSTALL_OPTIONAL_BOOSTING:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm', 'catboost', 'xgboost'], check=True)

## 2. Pobranie HYPERVIEW2 przez EOTDL

Pierwsze uruchomienie moze wymagac logowania. Jesli `eotdl auth login` poprosi o token lub otwarcie linku, wykonaj instrukcje z outputu komorki.

In [ ]:
from pathlib import Path

DATA_PARENT = Path('/content/data/hyperview2')
EXPECTED_HV2_ROOT = DATA_PARENT / 'HYPERVIEW2'
DATA_PARENT.mkdir(parents=True, exist_ok=True)


def is_hyperview2_root(path: Path) -> bool:
    return (
        (path / 'train_gt.csv').is_file()
        and (path / 'submission.csv').is_file()
        and (path / 'train' / 'hsi_satellite').is_dir()
        and (path / 'test' / 'hsi_satellite').is_dir()
    )


def find_hyperview2_root(search_root: Path) -> Path | None:
    candidates = [EXPECTED_HV2_ROOT, search_root]
    if search_root.exists():
        candidates.extend(path.parent for path in search_root.rglob('train_gt.csv'))
    for candidate in dict.fromkeys(candidates):
        if is_hyperview2_root(candidate):
            return candidate
    return None


HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
print('Dataset parent:', DATA_PARENT)
print('Dataset root:', HV2_ROOT)
print('Root ready:', is_hyperview2_root(HV2_ROOT))

In [ ]:
# Uruchom tylko jesli nie jestes jeszcze zalogowany w EOTDL.
!eotdl auth login

In [ ]:
import os
import subprocess

# Set to True if you want to redownload even when a valid dataset is already present.
FORCE_EOTDL_DOWNLOAD = False

HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
needs_download = not is_hyperview2_root(HV2_ROOT)
partial_expected_dir = EXPECTED_HV2_ROOT.exists() and not is_hyperview2_root(EXPECTED_HV2_ROOT)

if needs_download or FORCE_EOTDL_DOWNLOAD:
    env = os.environ.copy()
    env['EOTDL_STAGE_WORKERS'] = '16'
    cmd = [
        'eotdl', 'datasets', 'get', 'HYPERVIEW2',
        '--version', '2', '--assets', '--path', str(DATA_PARENT), '--verbose',
    ]
    if FORCE_EOTDL_DOWNLOAD or partial_expected_dir:
        cmd.append('--force')
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True, env=env)
    HV2_ROOT = find_hyperview2_root(DATA_PARENT) or EXPECTED_HV2_ROOT
else:
    print('HYPERVIEW2 already exists, skipping download.')

print('Dataset root:', HV2_ROOT)
print('Root ready:', is_hyperview2_root(HV2_ROOT))

## 3. Walidacja layoutu danych

In [ ]:
import os
from pathlib import Path

HV2_ROOT = find_hyperview2_root(DATA_PARENT) or HV2_ROOT
required = [
    HV2_ROOT / 'train_gt.csv',
    HV2_ROOT / 'submission.csv',
    HV2_ROOT / 'train' / 'hsi_satellite',
    HV2_ROOT / 'test' / 'hsi_satellite',
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    print('Current DATA_PARENT tree:')
    for path in sorted(DATA_PARENT.rglob('*'))[:80]:
        print(' ', path.relative_to(DATA_PARENT))
    raise FileNotFoundError(
        'Missing HYPERVIEW2 files/directories after EOTDL download. '
        'If DATA_PARENT contains a partial HYPERVIEW2 directory, rerun the download cell; '
        'it now adds --force automatically for partial downloads. Missing: '
        + ', '.join(missing)
    )

print('Resolved HYPERVIEW2 root:', HV2_ROOT)
for rel in ['train/hsi_satellite', 'train/hsi_airborne', 'train/msi_satellite', 'test/hsi_satellite', 'test/msi_satellite']:
    directory = HV2_ROOT / rel
    count = len(list(directory.glob('*.npz'))) if directory.exists() else 0
    print(f'{rel:24s} {count:5d} npz files')

## 4. Kod datasetu, cech i metryk

Ten blok jest samowystarczalny: nie importuje kodu z repo, zeby notebook dzialal bez pushowania najnowszego WIP.

In [ ]:
from __future__ import annotations

import csv
import json
import math
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

TARGET_COLUMNS = ('B', 'Cu', 'Zn', 'Fe', 'S', 'Mn')
MODALITY_DIRS = {
    'prisma': 'hsi_satellite',
    'airborne': 'hsi_airborne',
    'sentinel2': 'msi_satellite',
}
EXPECTED_BANDS = {
    'prisma': 230,
    'airborne': 430,
    'sentinel2': 13,
}
FEATURE_SETS = ('mean_std', 'mean_std_derivatives', 'full_stats')


@dataclass(frozen=True)
class Hyperview2Sample:
    sample_id: str
    array_path: Path
    target: np.ndarray


def canonical_sample_id(value: Any) -> str:
    text = str(value).strip()
    if not text:
        raise ValueError('Empty sample id')
    if text.isdigit():
        return str(int(text))
    return Path(text).stem


def array_stem_candidates(raw_id: Any) -> list[str]:
    sample_id = canonical_sample_id(raw_id)
    stems = [sample_id]
    if sample_id.isdigit():
        stems.insert(0, f'{int(sample_id):04d}')
    return list(dict.fromkeys(stems))


def resolve_array_path(array_dir: Path, raw_id: Any) -> Path:
    for stem in array_stem_candidates(raw_id):
        for suffix in ('.npz', '.npy'):
            path = array_dir / f'{stem}{suffix}'
            if path.is_file():
                return path
    raise FileNotFoundError(f'Could not resolve array for id={raw_id!r} in {array_dir}')


def build_samples(root: Path, modality: str = 'prisma', split: str = 'train') -> list[Hyperview2Sample]:
    if modality not in MODALITY_DIRS:
        raise ValueError(f'Unsupported modality: {modality}')
    if split != 'train':
        raise ValueError('Only train split has labels in this notebook protocol.')
    labels_csv = root / 'train_gt.csv'
    array_dir = root / split / MODALITY_DIRS[modality]
    if not labels_csv.is_file():
        raise FileNotFoundError(labels_csv)
    if not array_dir.is_dir():
        raise FileNotFoundError(array_dir)

    rows = pd.read_csv(labels_csv)
    if 'sample_index' not in rows.columns:
        raise ValueError('Expected sample_index column in train_gt.csv')
    missing_targets = [target for target in TARGET_COLUMNS if target not in rows.columns]
    if missing_targets:
        raise ValueError(f'Missing target columns: {missing_targets}')

    samples = []
    missing = []
    for _, row in rows.iterrows():
        raw_id = row['sample_index']
        try:
            array_path = resolve_array_path(array_dir, raw_id)
        except FileNotFoundError:
            missing.append(raw_id)
            continue
        target = row.loc[list(TARGET_COLUMNS)].to_numpy(dtype=np.float32)
        samples.append(Hyperview2Sample(canonical_sample_id(raw_id), array_path, target))
    if not samples:
        raise RuntimeError(f'No samples paired. Missing examples: {missing[:8]}')
    return sorted(samples, key=lambda sample: int(sample.sample_id) if sample.sample_id.isdigit() else sample.sample_id)


def to_chw(array: np.ndarray, expected_bands: int | None = None) -> np.ndarray:
    array = np.asarray(array)
    while array.ndim > 3:
        singleton_axes = [axis for axis, dim in enumerate(array.shape) if dim == 1]
        if not singleton_axes:
            break
        array = np.squeeze(array, axis=singleton_axes[0])
    if array.ndim != 3:
        raise ValueError(f'Expected 3D array, got {array.shape}')
    shape = tuple(int(dim) for dim in array.shape)
    axes = [axis for axis, dim in enumerate(shape) if expected_bands is not None and dim == expected_bands]
    if not axes:
        axes = [0] if shape[0] <= shape[-1] else [2]
    band_axis = axes[0]
    if band_axis == 0:
        chw = array
    elif band_axis == 2:
        chw = np.moveaxis(array, 2, 0)
    else:
        chw = np.moveaxis(array, band_axis, 0)
    return np.ascontiguousarray(chw, dtype=np.float32)


def load_cube_and_mask(path: Path, modality: str = 'prisma') -> tuple[np.ndarray, np.ndarray | None]:
    expected = EXPECTED_BANDS.get(modality)
    if path.suffix == '.npz':
        with np.load(path) as archive:
            data_key = 'data' if 'data' in archive.files else max(archive.files, key=lambda key: archive[key].size)
            cube = to_chw(archive[data_key], expected_bands=expected)
            raw_mask = archive['mask'] if 'mask' in archive.files else None
    else:
        cube = to_chw(np.load(path), expected_bands=expected)
        raw_mask = None
    mask = None
    if raw_mask is not None:
        raw_mask = np.asarray(raw_mask)
        raw_mask = to_chw(raw_mask, expected_bands=expected) if raw_mask.ndim == 3 else np.asarray(raw_mask)
        if raw_mask.ndim == 3:
            mask = raw_mask.astype(bool).max(axis=0)
        else:
            mask = np.squeeze(raw_mask).astype(bool)
        if mask.shape != cube.shape[-2:]:
            raise ValueError(f'Mask shape {mask.shape} does not match cube shape {cube.shape}')
    return cube, mask


def normalize_cube(cube: np.ndarray, mask: np.ndarray | None, mode: str = 'none') -> np.ndarray:
    cube = np.asarray(cube, dtype=np.float32)
    finite = np.isfinite(cube)
    if mask is not None:
        finite &= mask[None]
    values = cube[finite]
    if values.size == 0 or mode == 'none':
        return np.nan_to_num(cube, nan=0.0, posinf=0.0, neginf=0.0)
    if mode == 'minmax':
        low, high = float(values.min()), float(values.max())
    elif mode == 'percentile':
        low, high = np.percentile(values, [1.0, 99.0]).astype(np.float32)
        low, high = float(low), float(high)
    else:
        raise ValueError(f'Unsupported normalization: {mode}')
    if not math.isfinite(high - low) or high <= low:
        return np.zeros_like(cube, dtype=np.float32)
    return np.clip((cube - low) / (high - low), 0.0, 1.0).astype(np.float32)


def band_stat(values: np.ndarray, channels: int, statistic: str) -> np.ndarray:
    if values.size == 0:
        return np.zeros(channels, dtype=np.float32)
    if statistic == 'mean':
        return values.mean(axis=1).astype(np.float32)
    if statistic == 'std':
        return values.std(axis=1).astype(np.float32)
    if statistic == 'min':
        return values.min(axis=1).astype(np.float32)
    if statistic == 'max':
        return values.max(axis=1).astype(np.float32)
    if statistic == 'median':
        return np.median(values, axis=1).astype(np.float32)
    if statistic == 'q25':
        return np.percentile(values, 25.0, axis=1).astype(np.float32)
    if statistic == 'q75':
        return np.percentile(values, 75.0, axis=1).astype(np.float32)
    raise ValueError(statistic)


def spectral_gradient(values: np.ndarray) -> np.ndarray:
    if values.shape[0] <= 1:
        return np.zeros_like(values, dtype=np.float32)
    return np.gradient(values.astype(np.float32)).astype(np.float32)


def extract_features(cube: np.ndarray, mask: np.ndarray | None, normalization: str, feature_set: str) -> np.ndarray:
    if feature_set not in FEATURE_SETS:
        raise ValueError(f'feature_set must be one of {FEATURE_SETS}')
    cube = normalize_cube(cube, mask, mode=normalization)
    c, h, w = cube.shape
    flat = cube.reshape(c, h * w)
    valid = mask.reshape(h * w).astype(bool) if mask is not None else np.isfinite(flat).all(axis=0)
    valid_fraction = np.asarray([float(valid.mean()) if valid.size else 0.0], dtype=np.float32)
    valid_values = flat[:, valid] if valid.any() else np.empty((c, 0), dtype=np.float32)
    mean = band_stat(valid_values, c, 'mean')
    std = band_stat(valid_values, c, 'std')
    features = [mean, std]
    if feature_set in {'mean_std_derivatives', 'full_stats'}:
        features.extend([spectral_gradient(mean), spectral_gradient(std)])
    if feature_set == 'full_stats':
        features.extend([
            band_stat(valid_values, c, 'min'),
            band_stat(valid_values, c, 'max'),
            band_stat(valid_values, c, 'median'),
            band_stat(valid_values, c, 'q25'),
            band_stat(valid_values, c, 'q75'),
        ])
    features.append(valid_fraction)
    return np.concatenate(features).astype(np.float32)


def fixed_split(samples: list[Hyperview2Sample], val_fraction: float = 0.2, seed: int = 42):
    rng = np.random.default_rng(seed)
    indices = np.arange(len(samples))
    rng.shuffle(indices)
    val_count = max(1, int(round(len(samples) * val_fraction)))
    val_indices = set(indices[:val_count].tolist())
    train = [sample for idx, sample in enumerate(samples) if idx not in val_indices]
    val = [sample for idx, sample in enumerate(samples) if idx in val_indices]
    return train, val


def make_feature_matrix(samples: list[Hyperview2Sample], modality: str, normalization: str, feature_set: str):
    x_rows, y_rows, ids = [], [], []
    for sample in tqdm(samples, desc=f'features:{modality}:{feature_set}'):
        cube, mask = load_cube_and_mask(sample.array_path, modality=modality)
        x_rows.append(extract_features(cube, mask, normalization, feature_set))
        y_rows.append(sample.target)
        ids.append(sample.sample_id)
    return np.stack(x_rows).astype(np.float32), np.stack(y_rows).astype(np.float32), ids


def hyperview_score(y_true: np.ndarray, y_pred: np.ndarray, baseline_mse: np.ndarray, eps: float = 1e-12):
    mse = ((y_pred - y_true) ** 2).mean(axis=0)
    per_target = mse / np.maximum(baseline_mse, eps)
    return float(per_target.mean()), per_target.astype(np.float32)


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray, baseline_mse: np.ndarray) -> dict[str, Any]:
    mse = ((y_pred - y_true) ** 2).mean(axis=0)
    mae = np.abs(y_pred - y_true).mean(axis=0)
    score, relative_mse = hyperview_score(y_true, y_pred, baseline_mse)
    return {
        'hyperview_score': score,
        'mean_mse': float(mse.mean()),
        'mean_mae': float(mae.mean()),
        'targets': {
            target: {
                'mse': float(mse[idx]),
                'mae': float(mae[idx]),
                'rmse': float(np.sqrt(mse[idx])),
                'relative_mse': float(relative_mse[idx]),
                'baseline_mse': float(baseline_mse[idx]),
            }
            for idx, target in enumerate(TARGET_COLUMNS)
        },
    }


def json_ready(value: Any) -> Any:
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value

## 5. Gotowe regresory

In [ ]:
import importlib.util

from sklearn.cross_decomposition import PLSRegression
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import RidgeCV
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

REGRESSOR_DESCRIPTIONS = {
    'dummy_mean': 'Train-mean sanity baseline.',
    'ridge': 'Scaled RidgeCV multi-output linear baseline.',
    'pls': 'Partial Least Squares, standard chemometric spectral baseline.',
    'knn': 'Scaled KNN with distance weighting.',
    'extra_trees': 'Extremely randomized trees, close to public HYPERVIEW2 winning style.',
    'random_forest': 'Random forest HSI tabular baseline.',
    'hist_gradient_boosting': 'Sklearn histogram gradient boosting via MultiOutputRegressor.',
    'lightgbm': 'Optional LightGBM via MultiOutputRegressor.',
    'catboost': 'Optional CatBoost via MultiOutputRegressor.',
    'xgboost': 'Optional XGBoost via MultiOutputRegressor.',
}


def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None


def bounded_pls_components(requested: int, x_train: np.ndarray, y_train: np.ndarray) -> int:
    return max(1, min(requested, x_train.shape[1], x_train.shape[0] - 1, y_train.shape[1]))


def build_regressor(name: str, x_train: np.ndarray, y_train: np.ndarray, seed: int = 42, n_jobs: int = -1):
    if name == 'dummy_mean':
        return DummyRegressor(strategy='mean')
    if name == 'ridge':
        return make_pipeline(StandardScaler(), RidgeCV(alphas=(0.01, 0.1, 1.0, 10.0, 100.0)))
    if name == 'pls':
        return PLSRegression(n_components=bounded_pls_components(16, x_train, y_train), scale=True)
    if name == 'knn':
        return make_pipeline(StandardScaler(), KNeighborsRegressor(n_neighbors=8, weights='distance', n_jobs=n_jobs))
    if name == 'extra_trees':
        return ExtraTreesRegressor(n_estimators=600, max_features='sqrt', min_samples_leaf=2, random_state=seed, n_jobs=n_jobs)
    if name == 'random_forest':
        return RandomForestRegressor(n_estimators=500, max_features='sqrt', min_samples_leaf=2, random_state=seed, n_jobs=n_jobs)
    if name == 'hist_gradient_boosting':
        base = HistGradientBoostingRegressor(max_iter=400, learning_rate=0.05, random_state=seed)
        return MultiOutputRegressor(base, n_jobs=n_jobs)
    if name == 'lightgbm':
        if not has_module('lightgbm'):
            raise ImportError('Install lightgbm first or remove this model from MODEL_NAMES.')
        from lightgbm import LGBMRegressor
        base = LGBMRegressor(n_estimators=800, learning_rate=0.03, num_leaves=31, random_state=seed, n_jobs=n_jobs)
        return MultiOutputRegressor(base, n_jobs=n_jobs)
    if name == 'catboost':
        if not has_module('catboost'):
            raise ImportError('Install catboost first or remove this model from MODEL_NAMES.')
        from catboost import CatBoostRegressor
        base = CatBoostRegressor(iterations=800, learning_rate=0.03, depth=6, loss_function='RMSE', random_seed=seed, verbose=False, allow_writing_files=False)
        return MultiOutputRegressor(base, n_jobs=n_jobs)
    if name == 'xgboost':
        if not has_module('xgboost'):
            raise ImportError('Install xgboost first or remove this model from MODEL_NAMES.')
        from xgboost import XGBRegressor
        base = XGBRegressor(n_estimators=800, learning_rate=0.03, max_depth=6, objective='reg:squarederror', tree_method='hist', random_state=seed, n_jobs=n_jobs)
        return MultiOutputRegressor(base, n_jobs=n_jobs)
    raise ValueError(f'Unknown regressor: {name}')

## 6. Konfiguracja eksperymentu

`normalization='none'` zachowuje skale zapisane w HYPERVIEW2. Dla analizy odpornosci mozna tez uruchomic `percentile`, ale wtedy trzeba raportowac to jako osobny wariant preprocessingowy.

In [ ]:
MODALITY = 'prisma'
FEATURE_SET = 'mean_std_derivatives'
NORMALIZATION = 'none'
VAL_FRACTION = 0.2
SEED = 42
N_JOBS = -1

MODEL_NAMES = [
    'dummy_mean',
    'ridge',
    'pls',
    'knn',
    'extra_trees',
    'random_forest',
    'hist_gradient_boosting',
    # 'lightgbm',
    # 'catboost',
    # 'xgboost',
]

# Pozniej mozna tu dodac rekonstrukcje z kompresorow w takim samym canonical layoucie.
# Przyklad: {'mamba_k4_recon': Path('/content/reconstructions/mamba_k4/HYPERVIEW2')}
RECON_VARIANTS: dict[str, Path] = {}

OUTPUT_DIR = Path('/content/artifacts/downstream/hyperview2_regressors')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 7. Ekstrakcja cech z oryginalnych danych

In [ ]:
samples = build_samples(HV2_ROOT, modality=MODALITY, split='train')
train_samples, val_samples = fixed_split(samples, val_fraction=VAL_FRACTION, seed=SEED)

print('Samples:', len(samples))
print('Train:', len(train_samples), 'Val:', len(val_samples))
print('First train id:', train_samples[0].sample_id, 'First val id:', val_samples[0].sample_id)

In [ ]:
x_train, y_train, train_ids = make_feature_matrix(train_samples, MODALITY, NORMALIZATION, FEATURE_SET)
x_val, y_val, val_ids = make_feature_matrix(val_samples, MODALITY, NORMALIZATION, FEATURE_SET)

baseline_mse = ((y_val - y_train.mean(axis=0, keepdims=True)) ** 2).mean(axis=0).astype(np.float32)
print('X train:', x_train.shape, 'X val:', x_val.shape)
print('Baseline MSE:', dict(zip(TARGET_COLUMNS, baseline_mse.tolist())))

## 8. Trenowanie i ewaluacja regresorow na oryginalnych danych

In [ ]:
def run_regressors(x_train, y_train, x_val, y_val, baseline_mse, model_names):
    rows = []
    details = {}
    for name in model_names:
        start = time.perf_counter()
        row = {'model': name}
        try:
            model = build_regressor(name, x_train, y_train, seed=SEED, n_jobs=N_JOBS)
            model.fit(x_train, y_train)
            fit_time = time.perf_counter() - start
            pred_start = time.perf_counter()
            y_pred = np.asarray(model.predict(x_val), dtype=np.float32)
            pred_time = time.perf_counter() - pred_start
            metrics = regression_metrics(y_val, y_pred, baseline_mse)
            row.update({
                'status': 'ok',
                'hyperview_score': metrics['hyperview_score'],
                'mean_mse': metrics['mean_mse'],
                'mean_mae': metrics['mean_mae'],
                'fit_time_sec': fit_time,
                'predict_time_sec': pred_time,
            })
            for target, target_metrics in metrics['targets'].items():
                row[f'{target}_rmse'] = target_metrics['rmse']
                row[f'{target}_relative_mse'] = target_metrics['relative_mse']
            details[name] = {
                'status': 'ok',
                'description': REGRESSOR_DESCRIPTIONS.get(name),
                'metrics': metrics,
                'fit_time_sec': fit_time,
                'predict_time_sec': pred_time,
            }
        except Exception as exc:
            row.update({'status': 'failed', 'error': str(exc), 'fit_time_sec': time.perf_counter() - start})
            details[name] = {'status': 'failed', 'error': str(exc)}
        rows.append(row)
    df = pd.DataFrame(rows)
    if 'hyperview_score' in df.columns:
        df = df.sort_values(['status', 'hyperview_score'], na_position='last')
    return df, details


original_df, original_details = run_regressors(x_train, y_train, x_val, y_val, baseline_mse, MODEL_NAMES)
original_df

## 9. Opcjonalna ewaluacja rekonstrukcji z modeli kompresji

Gdy wygenerujemy rekonstrukcje z Mamby/baseline'ow, zapisujemy je w layoutcie zgodnym z EOTDL, np. `/content/reconstructions/mamba_k4/HYPERVIEW2/train/hsi_satellite/0000.npz`. Wtedy uzupelniamy `RECON_VARIANTS` i uruchamiamy te komorke.

Raportowane tryby:
- `original_train_to_recon_val`: regresor trenowany na oryginalach, walidowany na rekonstrukcjach; mierzy drop-in degradation,
- `recon_train_to_recon_val`: regresor trenowany i walidowany na rekonstrukcjach; mierzy wynik po dostosowaniu downstream do kompresji.

In [ ]:
reconstruction_results = {}

for variant_name, variant_root in RECON_VARIANTS.items():
    print('Variant:', variant_name, 'root:', variant_root)
    recon_samples = build_samples(Path(variant_root), modality=MODALITY, split='train')
    recon_by_id = {sample.sample_id: sample for sample in recon_samples}
    recon_train = [recon_by_id[sample.sample_id] for sample in train_samples]
    recon_val = [recon_by_id[sample.sample_id] for sample in val_samples]

    x_recon_train, y_recon_train, _ = make_feature_matrix(recon_train, MODALITY, NORMALIZATION, FEATURE_SET)
    x_recon_val, y_recon_val, _ = make_feature_matrix(recon_val, MODALITY, NORMALIZATION, FEATURE_SET)

    dropin_df, dropin_details = run_regressors(x_train, y_train, x_recon_val, y_recon_val, baseline_mse, MODEL_NAMES)
    retrain_df, retrain_details = run_regressors(x_recon_train, y_recon_train, x_recon_val, y_recon_val, baseline_mse, MODEL_NAMES)

    reconstruction_results[variant_name] = {
        'original_train_to_recon_val': dropin_details,
        'recon_train_to_recon_val': retrain_details,
        'original_train_to_recon_val_table': dropin_df.to_dict(orient='records'),
        'recon_train_to_recon_val_table': retrain_df.to_dict(orient='records'),
    }
    display(dropin_df)
    display(retrain_df)

if not RECON_VARIANTS:
    print('No reconstruction variants configured; skipping compression-aware downstream checks.')

## 10. Zapis wynikow

In [ ]:
summary_csv = OUTPUT_DIR / 'summary_original.csv'
metrics_json = OUTPUT_DIR / 'metrics.json'

protocol = {
    'dataset': 'HYPERVIEW2',
    'dataset_root': str(HV2_ROOT),
    'source': 'EOTDL HYPERVIEW2 version 2 assets',
    'split_source': 'train_gt.csv fixed internal train/validation split',
    'modality': MODALITY,
    'feature_set': FEATURE_SET,
    'normalization': NORMALIZATION,
    'val_fraction': VAL_FRACTION,
    'seed': SEED,
    'target_columns': list(TARGET_COLUMNS),
    'train_samples': len(train_samples),
    'val_samples': len(val_samples),
    'train_sample_ids': train_ids,
    'val_sample_ids': val_ids,
}

payload = {
    'protocol': protocol,
    'baseline_mse': baseline_mse.tolist(),
    'original': original_details,
    'reconstruction_variants': reconstruction_results,
}

original_df.to_csv(summary_csv, index=False)
metrics_json.write_text(json.dumps(json_ready(payload), indent=2, sort_keys=True), encoding='utf-8')

print('Saved:', summary_csv)
print('Saved:', metrics_json)

In [ ]:
!find /content/artifacts/downstream/hyperview2_regressors -maxdepth 2 -type f -print